# BUS-BRA: classification under lesion-boundary uncertainty (submission notebook)

Mostafa Bdeir and Hadi Fardan. This notebook walks through the whole workflow: data audit and patient splits, feature extraction, the Logistic Regression and RBF SVM baselines with nested tuning, the boundary-edit stress tests, stability feature selection, patient-level calibration, the saved held-out results, failure analysis, complexity analysis and loading the trained model.

**Three run modes** (set `RUN_MODE` in the next cell):

| Mode | What it does | Needs | Time |
|---|---|---|---|
| `saved` (default) | Shows the saved result tables, recomputes held-out metrics from saved patient probabilities, verifies and loads the trained model | This repository only | seconds |
| `derived` | Also retrains everything from the saved derived feature files (`derived_features/`): baselines, stress tests, stability selection, calibration | This repository only | about 3 minutes |
| `full` | Starts from the raw archive: audits it, extracts features and perturbed features, then runs every stage | `data/raw/BUSBRA.zip` ([Zenodo](https://zenodo.org/records/8231412)) | about 8 minutes |

**Held-out safety.** No mode runs the locked held-out evaluation (`src.heldout_evaluation`) or the final-report script. The held-out numbers here are recomputed from the saved patient probabilities only, and the notebook asserts this at the end. The development stages write to a temporary working copy, never to the repository.

## 1. Setup

In [ ]:
import os, sys, json, hashlib, shutil, subprocess, inspect, time
from pathlib import Path
from importlib.metadata import version
from tempfile import mkdtemp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, brier_score_loss, log_loss, confusion_matrix

RUN_MODE = "saved"          # "saved" | "derived" | "full"   <-- change this to retrain
RUN_BENCHMARK = False       # development-only timing benchmark (about 1 minute); needs RUN_MODE != "saved"

# Locate the repository root (the folder that contains src/features.py)
here = Path.cwd().resolve()
root = Path(os.environ.get("BUS_BRA_ROOT", next((p for p in (here, *here.parents) if (p / "src/features.py").is_file()), here))).resolve()
sys.path.insert(0, str(root))
tables = root / "reports/submission_v2/tables"
published = root / "reports/final_step22_run2"
derived = root / "derived_features"
archive = root / "data/raw/BUSBRA.zip"
assert RUN_MODE in ("saved", "derived", "full")
if RUN_MODE == "full" and not archive.is_file():
    raise FileNotFoundError(f"RUN_MODE='full' needs {archive} (download BUSBRA.zip from https://zenodo.org/records/8231412)")

pinned = {"numpy": "2.5.4", "pandas": "3.0.6", "scikit-learn": "1.9.1", "scipy": "1.18.1", "scikit-image": "0.26.0", "joblib": "1.6.0"}
installed = {p: version(p) for p in pinned}
print("Python", sys.version.split()[0], "| mode:", RUN_MODE)
for p, v in pinned.items():
    print(f"  {p:14s} installed {installed[p]:8s} pinned {v}", "" if installed[p] == v else "  <-- differs from the original run")

def show_table(name):
    "Display a saved result table from reports/submission_v2/tables."
    df = pd.read_csv(tables / name)
    display(df)
    return df

## 2. Data audit and patient-level splits

Every split is by **patient**, never by image, because a patient can have several views. The committed split files are locked: `src.splits` refuses to overwrite them if the dataset disagrees. Development = 851 patients (1,514 images) with five locked outer folds; test = 213 patients (361 images), untouched until the one-shot evaluation.

In [ ]:
images = pd.read_csv(root / "splits/image_splits.csv")
patients = pd.read_csv(root / "splits/patient_splits.csv")

# Integrity checks: unique IDs, one split and one fold per patient, no patient in both development and test
assert images.ID.is_unique and patients.Case.is_unique
assert images.groupby("Case").split.nunique().max() == 1
assert images.groupby("Case").dev_fold.nunique().max() == 1
assert set(patients.loc[patients.split == "development", "Case"]).isdisjoint(patients.loc[patients.split == "test", "Case"])
display(patients.groupby(["split", "Pathology"]).agg(patients=("Case", "size"), images=("n_images", "sum")))
print("Development patients per outer fold:\n", patients[patients.split == "development"].groupby("dev_fold").size().to_string())

if RUN_MODE == "full":
    # Audit the raw archive and verify the locked splits against it (writes into a temp copy, not the repo)
    print("Archive audit and split verification run in the working copy below.")

## 3. Working copy for the development stages

The stages below run as separate processes (`python -m src.<module>`) in a temporary copy of `src/` and `splits/`, so the repository is never modified. In `derived` mode the saved derived feature files are copied in; in `full` mode the raw archive is copied in and the extraction stages run first.

In [ ]:
work = None
def run(module, *args):
    "Run one pipeline stage in the working copy and print the tail of its output."
    assert module not in ("src.heldout_evaluation", "src.final_report"), "held-out stages are never run here"
    t0 = time.time()
    proc = subprocess.run([sys.executable, "-m", module, *args], cwd=work, capture_output=True, text=True)
    tail = "\n".join(proc.stdout.strip().splitlines()[-4:])
    print(f"[{module} {' '.join(args)}] {time.time()-t0:.0f}s exit={proc.returncode}\n{tail}\n")
    if proc.returncode:
        print(proc.stderr[-1500:])
        raise RuntimeError(f"{module} failed")

if RUN_MODE != "saved":
    work = Path(mkdtemp(prefix="busbra-work-"))
    shutil.copytree(root / "src", work / "src", ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copytree(root / "splits", work / "splits")
    (work / "data/processed").mkdir(parents=True)
    (work / "results/robustness").mkdir(parents=True)
    print("Working copy:", work)
    if RUN_MODE == "full":
        (work / "data/raw").mkdir(parents=True)
        shutil.copy2(archive, work / "data/raw/BUSBRA.zip")
        run("src.data")
        run("src.splits")                                   # verifies the locked split files against the archive
        run("src.features", "--all-development")            # 23 clean features per development image
        run("src.perturbations", "--all-development")       # original eight-condition masks (geometry audit)
        run("src.perturbed_features", "--all-development")  # features for those masks
        run("src.coherent_perturbations", "--all-development")
        run("src.coherent_features", "--all-development")
    else:
        for name in ("development_features.csv", "development_perturbed_features.csv", "development_coherent_features.csv"):
            shutil.copy2(derived / name, work / "data/processed" / name)
        shutil.copy2(derived / "coherent_geometry_development.csv", work / "results/robustness/coherent_geometry_development.csv")
        print("Derived feature files copied in; extraction stages are skipped.")
else:
    print("RUN_MODE='saved': stages are not run; saved tables are shown instead.")

## 4. Features and the two boundary-edit constructions

Each image and its lesion mask give 23 handcrafted features: six shape, seven intensity and ten masked texture features. The figure compares the **original** edit construction (boundary pixels chosen by distance to the contour, pseudo-random tie-break, which can scatter isolated pixel changes) with the **coherent** construction (one connected patch near a contour anchor). Both target an overlap (IoU) of about 0.95 with the original mask.

In [ ]:
from src.features import read_pair, preprocess_pair, extract_features, FEATURE_COLUMNS
from src.perturbations import partial_boundary_change, mask_iou
from src.coherent_perturbations import coherent_boundary_change

if archive.is_file():
    from zipfile import ZipFile
    image_id = images.loc[images.split.eq("development"), "ID"].sort_values().iloc[3]
    with ZipFile(archive) as z:
        gray, mask = read_pair(z, image_id)
    print("Real development image:", image_id)
else:
    yy, xx = np.ogrid[:160, :200]
    mask = ((yy - 80) / 32) ** 2 + ((xx - 100) / 44) ** 2 <= 1
    gray = (120 + 40 * np.sin(xx / 7.0) + 20 * np.cos(yy / 5.0)).clip(0, 255).astype(np.uint8) * np.ones_like(yy, dtype=np.uint8)
    print("Raw archive not present: using a synthetic ellipse so the demo still runs.")
gray, mask = preprocess_pair(gray, mask)          # grayscale, longest side 256, nearest-neighbour mask
orig_erosion = partial_boundary_change(mask, 0.95, "erosion")
coh_erosion = coherent_boundary_change(mask, 0.95, "erosion")
print(f"IoU original construction {mask_iou(mask, orig_erosion):.3f} | coherent construction {mask_iou(mask, coh_erosion):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, m, title in zip(axes, [mask, orig_erosion, coh_erosion], ["Original mask", "Original-construction erosion", "Coherent erosion"]):
    ax.imshow(gray, cmap="gray"); ax.contour(m, levels=[.5], colors="orange", linewidths=1.2)
    ax.contour(mask, levels=[.5], colors="cyan", linewidths=.6, linestyles="dashed"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

demo = pd.DataFrame([extract_features(gray, mask), extract_features(gray, orig_erosion), extract_features(gray, coh_erosion)],
                    index=["clean", "original erosion", "coherent erosion"]).T
demo["abs change (orig)"] = (demo["original erosion"] - demo["clean"]).abs()
demo["abs change (coherent)"] = (demo["coherent erosion"] - demo["clean"]).abs()
display(demo.round(4))

## 5. Baselines: Logistic Regression and RBF SVM with nested tuning

Outer loop: five locked patient folds. Inner loop: three patient-stratified folds choose the hyperparameters by patient balanced accuracy (ties broken by AUC, then the simpler setting). Scaling and fitting use training folds only, and each patient contributes total weight one across their views. The next cell shows the tuning code itself.

In [ ]:
import src.models as M
print(inspect.getsource(M.patient_image_weights))
print(inspect.getsource(M.choose_settings))

In [ ]:
if RUN_MODE != "saved":
    run("src.models")      # Logistic Regression: nested CV, C in {0.01..100}, class balancing off/on
    run("src.svm_model")   # RBF SVM: nested CV, C in {0.1,1,10}, gamma in {scale,0.01,0.1}, balancing off/on
    base = {n: json.loads((work / f"results/modeling/{n}_nested_cv_summary.json").read_text()) for n in ("logreg", "svm")}
    print({n: {k: round(v[k], 4) for k in ("balanced_accuracy", "roc_auc")} for n, v in base.items()})
else:
    print("Saved result: LR BA 86.89% / AUC 0.9304 ; SVM BA 87.07% / AUC 0.9352 (development, original clean baselines)")
print("Hyperparameters chosen in each outer fold (saved table):")
_ = show_table("table_chosen_hyperparameters.csv")

## 6. Boundary-sensitivity stress tests (development data, models fixed)

The trained models and thresholds stay fixed, and only the lesion mask changes. Eight conditions: mild and moderate erosion and dilation, and four shifts. Then the same erosion and dilation conditions with the coherent construction.

In [ ]:
if RUN_MODE != "saved":
    run("src.robustness_eval")      # original construction, eight conditions
    run("src.coherent_eval")        # coherent construction, four conditions (post-hoc, labelled as such)
print("Original construction (development, paired changes vs each model's own clean baseline):")
orig = show_table("table_original_eight_conditions.csv")
print("Coherent construction:")
_ = show_table("table_coherent_development.csv")
display(Image(filename=str(root / "reports/submission_v2/figures/06_original_and_coherent.png")))

## 7. Stability feature selection (the mitigation experiment)

Inside every training partition, each feature's stability is its largest mean absolute change under the four coherent edits, divided by its clean standard deviation (lower is better). Subsets of 5, 10 and 15 features are candidates; the model grids are tuned with an objective that weights clean balanced accuracy and the four perturbed balanced accuracies equally. A matched all-23 comparator uses the same grids and objective.

In [ ]:
import src.stable_selection as SS
print(inspect.getsource(SS.rank_training_features)[:2600], "\n...")

In [ ]:
if RUN_MODE != "saved":
    run("src.stable_selection")     # freezes the design and audits patient-disjoint training scopes
    run("src.stable_evaluation")    # nested evaluation of the subset arm vs the matched all-23 arm
print("Matched nested development comparison (saved table):")
_ = show_table("clean_summary.csv")
print("Features selected by the stability procedure, number of outer folds (of 5):")
_ = show_table("table_stability_selection_frequency.csv")

## 8. Patient-level probability calibration and the final models

Each outer training pool is split into three patient folds. For each fold, a base model is tuned on the other two folds and a sigmoid is fitted on the reserved patients' clean scores, keeping tuning, fitting and calibration patients disjoint. The three members are averaged. The decision threshold is fixed at 0.5.

In [ ]:
if RUN_MODE != "saved":
    run("src.patient_calibration")   # calibrated nested CV + final development-only ensembles (no held-out access)
    for leftover in ("held_out_evaluation_started.json",):
        assert not (work / "results/robustness/calibration_step20" / leftover).exists()
print("Calibrated outer-fold development performance (saved table):")
_ = show_table("calibrated_clean_summary.csv")
print("Fold-to-fold variation (calibrated development predictions):")
_ = show_table("table_fold_variation.csv")

## 9. Reproduction check

When a mode that retrains is used, the rebuilt numbers must equal the published development numbers. In `saved` mode the same constants are checked against the saved tables.

In [ ]:
published_numbers = {
    "LR baseline BA": 0.8689, "SVM baseline BA": 0.8707,
    "LR all-23 clean BA": 0.8735, "LR subset clean BA": 0.8455, "SVM all-23 clean BA": 0.8707, "SVM subset clean BA": 0.8687,
    "LR calibrated BA": 0.8416, "SVM calibrated BA": 0.8473,
}
src_dir = work / "results/robustness" if work else None
def tbl(path_work, path_saved):
    return pd.read_csv((src_dir / path_work) if work else (tables / path_saved))
cs = tbl("selection_step19/clean_summary.csv", "clean_summary.csv")
cal = tbl("calibration_step20/calibrated_clean_summary.csv", "calibrated_clean_summary.csv")
get = lambda df, m, arm=None: float(df[(df.model == m) & ((df.arm == arm) if arm else True)].balanced_accuracy.iloc[0])
observed = {
    # In retraining modes the baseline comes from the run just made; in saved mode from the saved summaries
    "LR baseline BA": base["logreg"]["balanced_accuracy"] if work else float(json.loads((tables / "baseline_summary_logreg.json").read_text())["balanced_accuracy"]),
    "SVM baseline BA": base["svm"]["balanced_accuracy"] if work else float(json.loads((tables / "baseline_summary_svm.json").read_text())["balanced_accuracy"]),
    "LR all-23 clean BA": get(cs, "logreg", "all_23_matched_tuning"), "LR subset clean BA": get(cs, "logreg", "stability_topk"),
    "SVM all-23 clean BA": get(cs, "svm", "all_23_matched_tuning"), "SVM subset clean BA": get(cs, "svm", "stability_topk"),
    "LR calibrated BA": get(cal, "logreg"), "SVM calibrated BA": get(cal, "svm"),
}
check = pd.DataFrame({"published": published_numbers, "observed": {k: round(v, 4) for k, v in observed.items()}})
check["match"] = (check.published - check.observed).abs() < 5e-4
display(check)
assert check.match.all(), "a development number does not match the published one"
print("All development numbers match the published ones.")

## 10. Locked held-out results, recomputed from saved patient probabilities

The one-shot held-out evaluation was run once during the study with frozen models. Here nothing is refitted or re-predicted: the metrics are recomputed from the saved per-patient probabilities (`patient_failure_audit.csv`), and the paired comparison counts patients.

In [ ]:
audit = pd.read_csv(published / "patient_failure_audit.csv")
rows = []
for name in ("lr", "svm"):
    y, p = audit.actual, audit[f"{name}_probability"]
    tn, fp, fn, tp = confusion_matrix(y, p >= .5).ravel()
    rows.append({"model": name.upper(), "BA %": 100 * balanced_accuracy_score(y, p >= .5), "sensitivity %": 100 * tp / (tp + fn),
                 "specificity %": 100 * tn / (tn + fp), "AUC": roc_auc_score(y, p), "Brier": brier_score_loss(y, p),
                 "log loss": log_loss(y, p), "FN": fn, "FP": fp})
display(pd.DataFrame(rows).round(4))
both = int((audit.lr_error & audit.svm_error).sum()); lr_only = int((audit.lr_error & ~audit.svm_error).sum())
svm_only = int((~audit.lr_error & audit.svm_error).sum()); neither = int((~audit.lr_error & ~audit.svm_error).sum())
print(f"misclassified by both {both} | LR only {lr_only} | SVM only {svm_only} | correct for both {neither} | patients {len(audit)}")
assert (both, lr_only, svm_only, neither) == (30, 3, 0, 180)
print("Held-out coherent boundary sensitivity (saved condition table):")
display(pd.read_csv(published / "heldout_robustness_table.csv")[["model", "condition", "delta_balanced_accuracy", "flip_rate", "new_false_negatives", "recovered_false_negatives", "delta_roc_auc"]].round(4))

## 11. Failure analysis and complexity analysis (development data)

Errors by lesion size (tertiles of the mean area fraction of a patient's images) and by number of views, from calibrated out-of-fold predictions; and a timing comparison of one tuned LR fit against one tuned SVM fit on a development fold, with no tuning inside the timing.

In [ ]:
fb = pd.read_csv(tables / "table_failure_breakdown.csv")
display(fb.pivot_table(index=["Model", "Class", "Factor"], columns="Group", values="Error rate (%)", aggfunc="first"))
print("Complexity (single development fold; comparable single-model operations, not the three-member ensembles):")
_ = show_table("table_complexity.csv")
if RUN_BENCHMARK and RUN_MODE != "saved":
    shutil.copytree(root / "analysis", work / "analysis", dirs_exist_ok=True)
    proc = subprocess.run([sys.executable, "-m", "analysis.development_analysis", "--root", str(work), "--repeats", "10"],
                          cwd=work, capture_output=True, text=True)
    print(proc.stdout[-2500:] or proc.stderr[-1500:])

## 12. The trained model

`models/frozen_calibrated_models.joblib` holds the final three-member calibrated LR and SVM ensembles. The cell verifies its bytes and the library versions against the recorded protocol *before* deserialising it, then predicts for one **development** patient. (No test patients are used.)

In [ ]:
import joblib
from src.patient_calibration import predict_ensemble
from src.models import load_development

model_path = root / "models/frozen_calibrated_models.joblib"
protocol = json.loads((root / "models/final_evaluation_protocol.json").read_text())
summary = json.loads((root / "models/calibration_summary.json").read_text())
expected = "627ef9df92e4b12b475a12c844b53e82e9dd7fefe05a9f07c3b94ca7c77f3586"   # digest recorded in the original report manifest
with model_path.open("rb") as f:
    actual = hashlib.file_digest(f, "sha256").hexdigest()
assert actual == expected == protocol["model_artifact_sha256"], "model checksum mismatch"
print("SHA-256 matches the original frozen model:", actual)
mismatch = {p: (version(p), v) for p, v in summary["versions"].items() if version(p) != v}
print("Library versions differ from the recorded ones:", mismatch or "none")
if mismatch:
    print("(Loading may still work; install the pinned requirements to match exactly.)")

bundle = joblib.load(model_path)
dev = load_development(derived / "development_features.csv", root / "splits/image_splits.csv")
sample = dev.loc[dev.Case.eq(dev.Case.iloc[0])]
print("Development patient", int(sample.Case.iloc[0]), "| true label:", sample.Pathology.iloc[0], "| views:", len(sample))
for name, ensemble in bundle["models"].items():
    print(name, [m["settings"] for m in ensemble["members"]])
    display(predict_ensemble(ensemble, sample)[["actual", "n_images", "prob_malignant", "predicted"]])

## 13. Safety check and summary

The notebook never ran the locked held-out evaluation. Summary of what it showed:

- Both baselines and every later development stage reproduce the published development numbers exactly.
- On the 213 held-out patients the calibrated SVM reached balanced accuracy 81.85% against 80.42% for LR; the paired difference is small (interval reaches zero), LR has the better AUC and probability losses, and the SVM missed 20 of 68 malignant patients at the fixed 0.5 threshold.
- Under the original boundary-edit construction the SVM lost less accuracy than LR; under the coherent construction both lost little; stability selection did not help.

In [ ]:
# run() refuses the held-out modules; also confirm no held-out marker or output exists in the working copy
if work:
    assert not list(work.rglob("held_out_evaluation_started.json")), "a held-out marker was created"
    assert not (work / "results/robustness/heldout_step21").exists(), "held-out outputs exist"
print("No held-out stage was run. Mode used:", RUN_MODE)